## Import packages

In [1]:
# Import thư viện dùng để phân tích cảm xúc
import pandas as pd
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()
# Đọc dữ liệu từ file CSV
df = pd.read_csv('customer_review.csv')
df.head(10)

[nltk_data] Error loading vader_lexicon: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1020)>


,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText
0,1,77,18,23/12/23,3,"Average experience, nothing special."
1,2,80,19,25/12/24,5,The quality is top-notch.
2,3,50,13,26/1/25,4,Five stars for the quick delivery.
3,4,78,15,21/4/25,3,"Good quality, but could be cheaper."
4,5,64,2,16/7/23,3,"Average experience, nothing special."
5,6,81,1,21/12/25,4,Customer support was very helpful.
6,7,16,1,29/1/24,3,"Average experience, nothing special."
7,8,55,8,15/8/24,5,The quality is top-notch.
8,9,3,13,1/9/23,4,"I love this product, will buy again!"
9,10,78,6,17/6/24,5,"Excellent product, highly recommend!"


## Clean data

In [3]:
# 1. Clean ReviewText (xóa khoảng trắng thừa)
df['ReviewText'] = df['ReviewText'].str.strip() \
                                   .str.replace(r'\s+', ' ', regex=True)# Xoá khoảng trắng trong cột ReviewText

# 2. Convert ReviewDate → datetime chuẩn
df['ReviewDate'] = pd.to_datetime(df['ReviewDate'], dayfirst=True)
df.head()



,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText
0,1,77,18,2023-12-23,3,"Average experience, nothing special."
1,2,80,19,2024-12-25,5,The quality is top-notch.
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper."
4,5,64,2,2023-07-16,3,"Average experience, nothing special."


## Tạo 2 cột mới Sentiment score và Sentiment label

In [9]:

# Tạo file csv mới sau khi làm sạch dữ liệu
df.to_csv('clean_customer_review.csv', index=False)
df1 = pd.read_csv('clean_customer_review.csv')
# Phân loại dựa trên điểm sentiment và rating
def classify_sentiment(score,rating): 
        if score > 0.05:  # Lọc dựa vào sentiment và rating để phân loại
            if rating >= 4:
                return 'tích cực'  
            elif rating == 3:
                return 'khá tích cực'  
            else:
                return 'khá tiêu cực'  
        elif score < -0.05:  
            if rating <= 2:
                return 'tiêu cực'  
            elif rating == 3:
                return 'khá tiêu cực'  
            else:
                return 'khá tích cực'  
        else:  
            if rating >= 4:
                return 'tích cực'  
            elif rating <= 2:
                return 'tiêu cực'  
            else:
                return 'trung lập'  
# Nhóm điểm sentiment vào các bucket
def sentiment_bucket(score): 
        if score >= 0.5:
            return '0.5 to 1.0'  
        elif 0.0 <= score < 0.5:
            return '0.0 to 0.49' 
        elif -0.5 <= score < 0.0:
            return '-0.49 to 0.0'  
        else:
            return '-1.0 to -0.5'  
# Áp dụng hàm phân loại sentiment và nhóm bucket vào DataFrame
# Tạo cột sentiment score bằng cách sử dụng VADER để tính điểm sentiment từ ReviewText
df1['sentiment_score'] = df1['ReviewText'].apply(lambda x: sia.polarity_scores(x)['compound']) # Hàm trong Vader
# Tạo cột sentiment label dựa trên điểm sentiment và rating
df1['sentiment_label'] = df1.apply(
    lambda row: classify_sentiment(row['sentiment_score'], row['Rating']),
    axis=1)
# Tạo cột sentiment bucket dựa trên điểm sentiment
df1['sentiment_bucket'] = df1['sentiment_score'].apply(sentiment_bucket)
# Ghi lại kết quả phân tích vào file CSV
df1.to_csv('clean_customer_review.csv', index=False)
df1.head(10)

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,sentiment_score,sentiment_label,sentiment_bucket
0,1,77,18,2023-12-23,3,"Average experience, nothing special.",-0.3089,khá tiêu cực,-0.49 to 0.0
1,2,80,19,2024-12-25,5,The quality is top-notch.,0.0000,tích cực,0.0 to 0.49
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.,0.0000,tích cực,0.0 to 0.49
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper.",0.2382,khá tích cực,0.0 to 0.49
4,5,64,2,2023-07-16,3,"Average experience, nothing special.",-0.3089,khá tiêu cực,-0.49 to 0.0
5,6,81,1,2025-12-21,4,Customer support was very helpful.,0.6997,tích cực,0.5 to 1.0
6,7,16,1,2024-01-29,3,"Average experience, nothing special.",-0.3089,khá tiêu cực,-0.49 to 0.0
7,8,55,8,2024-08-15,5,The quality is top-notch.,0.0000,tích cực,0.0 to 0.49
8,9,3,13,2023-09-01,4,"I love this product, will buy again!",0.6696,tích cực,0.5 to 1.0
9,10,78,6,2024-06-17,5,"Excellent product, highly recommend!",0.7773,tích cực,0.5 to 1.0


## Lọc khách hàng có đánh giá tích cực

In [41]:
df1[df1['sentiment_label'] == 'tích cực'][[ 'ReviewID' , 'CustomerID','sentiment_label','sentiment_bucket']]

,ReviewID,CustomerID,sentiment_label,sentiment_bucket
1,2,80,tích cực,0.0 to 0.49
2,3,50,tích cực,0.0 to 0.49
5,6,81,tích cực,0.5 to 1.0
7,8,55,tích cực,0.0 to 0.49
8,9,3,tích cực,0.5 to 1.0
...,...,...,...,...
1352,1353,34,tích cực,0.0 to 0.49
1353,1354,53,tích cực,0.5 to 1.0
1354,1355,9,tích cực,0.5 to 1.0
1355,1356,14,tích cực,0.0 to 0.49
